In [1]:
!pip install transformers

ЗАДАНИЕ 1. Сравнение токенизаторов

Загрузите токенизатор для английского языка (например, bert-base-uncased) и для русского (DeepPavlov/rubert-base-cased). Токенизируйте одно и то же предложение на двух языках (например, "I study transformers" и его перевод). Сравните количество токенов, наличие подслов, специальные токены. Объясните, почему результаты различаются.

In [2]:
from transformers import AutoTokenizer

# Загружаем два токенизатора
tokenizer_en = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer_ru = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")

text_en = "I study transformers"
text_ru = "Я изучаю трансформеры"

# Токенизируем английский текст английской моделью
tokens_en = tokenizer_en.tokenize(text_en)
# Токенизируем русский текст русской моделью
tokens_ru = tokenizer_ru.tokenize(text_ru)

print("Английский токенизатор (англ. текст):", tokens_en)
print("Русский токенизатор (рус. текст):", tokens_ru)

# А теперь попробуем скормить русский текст английскому токенизатору
tokens_ru_bad = tokenizer_en.tokenize(text_ru)
print("\nАнглийский токенизатор (рус. текст):", tokens_ru_bad)

# Посмотрим на специальные токены (вызов самого токенизатора)
encoded_ru = tokenizer_ru(text_ru)
print("\nID токенов с добавлением спец. символов:", encoded_ru['input_ids'])
print("Восстановленные токены по ID:", tokenizer_ru.convert_ids_to_tokens(encoded_ru['input_ids']))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Английский токенизатор (англ. текст): ['i', 'study', 'transformers']
Русский токенизатор (рус. текст): ['Я', 'изуч', '##аю', 'трансформ', '##еры']

Английский токенизатор (рус. текст): ['я', 'и', '##з', '##у', '##ч', '##а', '##ю', 'т', '##р', '##ан', '##с', '##ф', '##о', '##р', '##м', '##е', '##р', '##ы']

ID токенов с добавлением спец. символов: [101, 839, 9324, 16988, 26228, 5977, 102]
Восстановленные токены по ID: ['[CLS]', 'Я', 'изуч', '##аю', 'трансформ', '##еры', '[SEP]']


Сравнение: Английская модель отлично разбивает английский текст на целые слова. Русская модель тоже справляется с русским текстом (слово "трансформеры" может быть разбито на корень и окончание с префиксом ##, если модель сочтет это редким словом, но DeepPavlov знает его целиком).

Почему результаты различаются: Английский токенизатор не имеет в своем словаре кириллических слов. Если дать ему русский текст, он разобьет его на бессмысленные одиночные буквы и символы (или выдаст [UNK] — unknown). У каждой модели свой словарь (Vocabulary), собранный на корпусе конкретного языка.

Специальные токены: При вызове токенизатора (а не метода .tokenize()) по краям автоматически добавляются служебные токены: [CLS] (ID 101) в начало и [SEP] (ID 102) в конец.

ЗАДАНИЕ 2. Влияние параметров padding и truncation

Возьмите несколько предложений разной длины. Токенизируйте их с параметрами padding=True, truncation=True, max_length=10. Выведите полученные input_ids и attention_mask. Проанализируйте, как ведёт себя паддинг (какой токен добавляется) и обрезание. Что произойдёт с очень коротким предложением? А с очень длинным?

In [3]:
sentences =[
    "Я сплю.", # Очень короткое
    "Нейронные сети и трансформеры стали основой для обработки естественного языка в современном мире искусственного интеллекта." # Очень длинное
]

# Токенизируем с жесткими лимитами
inputs = tokenizer_ru(
    sentences,
    padding=True,
    truncation=True,
    max_length=10,
    return_tensors="pt"
)

print("Input IDs:\n", inputs['input_ids'])
print("\nAttention Mask:\n", inputs['attention_mask'])
for i, seq in enumerate(inputs['input_ids']):
    print(f"\nВосстановленный текст {i+1}: {tokenizer_ru.decode(seq)}")

Input IDs:
 tensor([[  101,   839, 52440,   898,   132,   102,     0,     0,     0,     0],
        [  101, 29503,  4102,  2059, 13148,   851, 26228,  5977,  7788,   102]])

Attention Mask:
 tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

Восстановленный текст 1: [CLS] Я сплю. [SEP] [PAD] [PAD] [PAD] [PAD]

Восстановленный текст 2: [CLS] Нейронные сети и трансформеры стали [SEP]


Паддинг: Короткое предложение выравнивается до заданной длины max_length=10 с помощью специального токена [PAD] (его ID = 0). В attention_mask на позициях паддинга стоят 0 — это подсказка нейросети "не обращай внимание на этот мусор, это пустышки".

Обрезание (Truncation): Длинное предложение жестко обрезается. В массиве остаётся ровно 10 токенов, включая обязательные [CLS] в начале и [SEP] в конце (то есть смысловых токенов влезает всего 8). Конец предложения просто теряется.

ЗАДАНИЕ 3. Исследование специальных токенов

Узнайте ID специальных токенов ([CLS], [SEP], [PAD], [MASK]) для выбранного токенизатора. Токенизируйте два предложения как пару (например, используя tokenizer с return_tensors='pt'). Посмотрите, где появляются эти токены и как меняется token_type_ids. Объясните их назначение.

In [4]:
print("ID специальных токенов:")
print(f"[CLS]: {tokenizer_ru.cls_token_id}")
print(f"[SEP]: {tokenizer_ru.sep_token_id}")
print(f"[PAD]: {tokenizer_ru.pad_token_id}")
print(f"[MASK]: {tokenizer_ru.mask_token_id}")

# Передаем два предложения как ПАРУ (через запятую)
pair_inputs = tokenizer_ru("Где моя стипендия?", "Она не пришла.", return_tensors="pt")

print("\nID токенов пары:", pair_inputs['input_ids'][0].tolist())
print("Token Type IDs:", pair_inputs['token_type_ids'][0].tolist())
print("Декодированный вид:", tokenizer_ru.decode(pair_inputs['input_ids'][0]))

ID специальных токенов:
[CLS]: 101
[SEP]: 102
[PAD]: 0
[MASK]: 103

ID токенов пары: [101, 38251, 35125, 64478, 878, 166, 102, 9698, 1699, 23263, 132, 102]
Token Type IDs: [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1]
Декодированный вид: [CLS] Где моя стипендия? [SEP] Она не пришла. [SEP]


Назначение специальных токенов:

[CLS] (101) — всегда стоит в начале. Вектор этого токена аккумулирует смысл всего предложения (используется для классификации текста).

[SEP] (102) — сепаратор. Отделяет предложения друг от друга и ставится в самом конце.

[PAD] (0) — заполнитель пустот (паддинг).

[MASK] (103) — используется при обучении модели угадывать пропущенные слова (Masked Language Modeling).

Token Type IDs: При передаче пары предложений этот массив указывает принадлежность: 0 значит, что токен принадлежит первому предложению, а 1 — второму. Это нужно для задач вроде "Вопрос-Ответ", чтобы модель понимала, где кончается контекст и начинается вопрос.


ЗАДАНИЕ 4. Декодирование и восстановление текста

Возьмите предложение, содержащее редкие слова, знаки препинания или цифры (например, "BERT — это модель 2024 года!"). Токенизируйте его, затем декодируйте полученные ID обратно в строку. Сравните исходный текст с декодированным. Объясните, какие изменения произошли и почему.

In [5]:
text_to_decode = "BERT — это супер-модель 2024 года!!!"

# Токенизируем
tokens_dict = tokenizer_ru(text_to_decode)

# Декодируем обратно
decoded_text = tokenizer_ru.decode(tokens_dict['input_ids'])

print("Оригинал:", text_to_decode)
print("Декодировано:", decoded_text)

Оригинал: BERT — это супер-модель 2024 года!!!
Декодировано: [CLS] BERT — это супер - модель 2024 года!!! [SEP]


Изменения:

По краям добавились [CLS] и [SEP].

Вокруг некоторых знаков препинания (например, тире или восклицательных знаков) могли появиться лишние пробелы.

Почему это произошло: При токенизации знаки препинания отделяются от слов и становятся самостоятельными токенами. Метод .decode() по умолчанию просто склеивает все токены через пробел. Чтобы убрать служебные токены при декодировании, нужно передавать аргумент skip_special_tokens=True.

ЗАДАНИЕ 5. Обработка неизвестных слов (OOV) Найдите слово, которого нет в словаре токенизатора (например, специальный термин или слово с ошибкой). Токенизируйте его и посмотрите, на какие подслова оно разбивается. Затем попробуйте декодировать эти подслова обратно. Сделай вывод о том, какой используется механизм подслов в выбранной модели (BPE, WordPiece)?

In [6]:
oov_word = "мегаультрагигачад"

# Смотрим именно на подслова
subwords = tokenizer_ru.tokenize(oov_word)
print("Разбиение на подслова:", subwords)

# Конвертируем в ID и декодируем обратно
ids = tokenizer_ru.convert_tokens_to_ids(subwords)
restored = tokenizer_ru.decode(ids)
print("Восстановленное слово:", restored)

Разбиение на подслова: ['мега', '##ульт', '##раг', '##ига', '##ча', '##д']
Восстановленное слово: мегаультрагигачад


OOV (Out Of Vocabulary): Модель не знает слова "мегаультрагигачад". Вместо того чтобы выдать ошибку или токен [UNK], она разбивает его на известные ей подслова (части).

Механизм подслов: Мы видим, что последующие куски слова начинаются с символов ## (например, ##ги, ##га, ##чад). Это явный признак алгоритма WordPiece (именно он используется в семействе BERT). Символы ## показывают токенизатору, что при обратном декодировании этот кусок нужно "приклеить" к предыдущему без пробела. Поэтому при вызове .decode() слово восстанавливается идеально, несмотря на то, что изначально его не было в словаре.